# Tiers 1 and 2: zero-shot language-model pair scores (GPU)

RUN_DECISIONS D19, "tiers 1 and 2 design". Needs `t12_bundle.zip` in `My Drive/fragmentation_t12/`. Use an A100 runtime. Re-running resumes: scored units are skipped. When it prints `T12 SCORING COMPLETE`, zip `out/` is not needed: the laptop reads `My Drive/fragmentation_t12/out/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile, subprocess, json
BASE = '/content/drive/MyDrive/fragmentation_t12'
OUT = BASE + '/out'
assert os.path.exists(BASE + '/t12_bundle.zip'), 't12_bundle.zip is not in ' + BASE
zipfile.ZipFile(BASE + '/t12_bundle.zip').extractall('/content/t12')
print(json.load(open('/content/t12/BUNDLE.json')))
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout or 'NO GPU')

In [ ]:
# smoke test: first 2 units, both tiers (timing only; full runs resume past them)
!cd /content/t12 && python -u runs/t12_score.py --tier T1 --out "$OUT" --limit 2
!cd /content/t12 && python -u runs/t12_score.py --tier T2 --out "$OUT" --limit 2

In [ ]:
!cd /content/t12 && python -u runs/t12_score.py --tier T1 --out "$OUT"

In [ ]:
!cd /content/t12 && python -u runs/t12_score.py --tier T2 --out "$OUT"

In [ ]:
for t in ('T1', 'T2'):
    r = json.load(open(f'{OUT}/{t}/SCORE_DONE.json'))
    print(t, {k: v for k, v in r.items() if k != 'manifest'})
print('T12 SCORING COMPLETE')

In [ ]:
# end the runtime (stops GPU billing). Runs only after both tiers finished.
for t in ('T1', 'T2'):
    assert os.path.exists(f'{OUT}/{t}/SCORE_DONE.json'), t + ' not finished: runtime left running'
drive.flush_and_unmount()
from google.colab import runtime
runtime.unassign()